# SuttaPlayer Piper1 VITS Control Panel (v15)
This notebook implements the decoupled, single-responsibility **Source of Truth** architecture for running your T4 GPU training runs. It is completely optimized for the Colab native terminal workspace with automated pre-flight gating and zero-trash telemetry.

### Step 1a: Mount Google Drive & Deno install

In [ ]:
# from google.colab import drive
drive.mount('/content/drive')

# Install Deno globally (takes ~2 seconds)
%curl -fsSL https://deno.land/install.sh | sh
import os
os.environ['PATH'] += ':/root/.deno/bin'

### Step 1b: Prepare Container (Git clone & manifests) 

In [ ]:
import os
%cd /content/drive/MyDrive

# Cleanly clone or pull origin changes without crashing
if not os.path.exists("sutta-tts-model-training"):
    !git clone https://github.com/dhamma-initiative/sutta-tts-model-training.git
else:
    print("📦 Repository folder exists. Fetching and pulling latest trial branches...")

# Force checkout and pull safely using -C (directory override)
!git -C sutta-tts-model-training fetch origin
!git -C sutta-tts-model-training checkout colab-trials
!git -C sutta-tts-model-training pull origin colab-trials

# Gdrive persistent checkpoints root
!mkdir -p piper_training/checkpoints
%cd /content

### Step 1b: CPU assets; extracts to /content/drive/MyDrive/piper_training

In [ ]:
%%writefile /content/drive/MyDrive/piper_training/training-cpu-stage-manifest.csv
resource,destination,extract_to,gdrive_id
en-gb_pisi-suttaplayer-medium-vits2023-dataset.tgz,/content/drive/MyDrive/piper_training,/content/drive/MyDrive/piper_training,1zec0dXEt8pIHqATGK0EIIbEDM2gB_H4M
last-9437-90576.ckpt,/content/drive/MyDrive/piper_training,,1oCCUYEyNdb1IOyisL-05pjtTF118qgQ6

In [ ]:
# IF FIRST TRAINING RUN run then source the resuming checkpoint from huggingface! (ie. NOT FROM training-cpu-stage-manifest.csv) 
!wget "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_GB/northern_english_male/medium/epoch%3D9029-step%3D2261720.ckpt?download=true" -O /content/drive/MyDrive/piper_training/last.ckpt

### Step 1c: GPU assets; extracts to /content

In [ ]:
# FAST-EXTRACT-BUILD IS UNDER REVIEW

# %%writefile /content/drive/MyDrive/piper_training/training-gpu-stage-manifest.csv
# resource,destination,extract_to,gdrive_id
# piper_cache.tgz,/content/drive/MyDrive/piper_training,/content,1LpOwLRVmjbgfcuuzWAGUhndK610UN3f8
# sutta_wheels_backup.tar.gz,/content/drive/MyDrive/piper_training,/content,1ANL7SEQQaSDGlM9S0evJM2FgM97z1tyE
# piper1_compiled_backup.tar.gz,/content/drive/MyDrive/piper_training,/content,11h_W4qCX9gYcwgY-Gx0F8CI_DlsxuOUK


### Step 2: Resource Acquisition & Environment Restore
Execute these two cells back-to-back. The first cell pulls large datasets, dependencies, and model states from shared GDrive IDs using shared manifests. The second cell performs a **100% offline pip install** from your local cache and links precompiled Cython/C++ folders locally in under 30 seconds.

In [ ]:
# Cell 2.1 [download, EXTRACT]: Pull and Stage assets uing CPU runtime
!deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-gdown-resources.ts \
  -i /content/drive/MyDrive/piper_training/training-cpu-stage-manifest.csv --extract

# UNDER REVIEW

# Cell 2.2a [download]: Pull and Stage assets for GPU runtime
# !deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-gdown-resources.ts \
#   -i /content/drive/MyDrive/piper_training/training-gpu-stage-manifest.csv

In [ ]:
!ln -s /content/drive/MyDrive/piper_training/last-9437-90576.ckpt /content/drive/MyDrive/piper_training/last.ckpt

### GPU
1. Shutdown CPU runtime
2. Runtime > Change runtime type
    * Change to [Runtime Type: "Python 3", Hardware acceleratorL "T4 GPU", Runtime Version: "2025.07"]
    * Save
3. Connect to Runtime
4. **Rerun Step 1a**: Mount Google Drive & Deno install

In [ ]:
# UNDER REVIEW

# Cell 2.2b [extract]: Pull and Stage assets for GPU runtime
# !deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-gdown-resources.ts \
#   -i /content/drive/MyDrive/piper_training/training-gpu-stage-manifest.csv --extract

### UNDER REVIEW Bootstrap patch

In [ ]:
# UNDER REVIEW

# %pip install scikit-build cmake ninja
# %pip install onnxruntime
# %pip install "pysilero-vad<3"
# %pip install pathvalidate tensorboardX

### UNDER REVIEW Finalise Bootstrap

In [ ]:
# UNDER REVIEW

# Cell 2.2: Rebuild PyTorch and link pre-compiled Cython/C++ libraries locally
# !deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-bootstrap.ts

### STREAMLINED BIG-BUILD [TO BE TESTED IN NEXT COLAB TRAINING]

In [ ]:
# 1. Target PyTorch/ONNX/Lightning stack
!pip install --no-cache-dir torch==2.3.1 lightning==2.3.3 onnx==1.15.0 onnxruntime-gpu

# 2. Clone repository
%cd /content
!rm -rf piper1-gpl
!git clone https://github.com/OHF-voice/piper1-gpl.git

# 3. Install Piper and all required training/CLI support dependencies
%cd /content/piper1-gpl
!pip install --no-deps -e ".[train]"
!pip install scikit-build setuptools cython librosa pydantic pytorch-lightning monotonic-align pysilero-vad pathvalidate "jsonargparse[signatures]>=4.27.7"

# 4. Build C++ / Cython extensions
!chmod +x ./build_monotonic_align.sh
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace

### Step 3: Pre-Flight Cross-Check ("Green for Go" Gating Engine)
To prevent duplicate efforts or unrecoverable crashes due to sudden GDrive FUSE mount drops, the `PreFlightValidator` acts as a strict proxy. It parses your actual `train_cmd` string, extracts target paths, runs physical sanity assertions, checks your checkpoint metadata (Epoch/Global Step), and **only unlocks the training loop if all checks are 100% green**.

In [ ]:
%ls /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta_training_validator.py

In [ ]:
# NOT PURSING FLAG:
# --model.c_dur 2.5

In [ ]:
import sys
sys.path.append("/content/drive/MyDrive/sutta-tts-model-training/scripts/")
from sutta_training_validator import PreFlightValidator   

train_cmd = """
PYTHONPATH=/content/drive/MyDrive/sutta-tts-model-training/scripts:$PYTHONPATH \
python3 -m piper.train fit \
  --data.voice_name "en_gb-suttaplayer-medium" \
  --data.csv_path "/content/drive/MyDrive/sutta-tts-model-training/corpus-preperation/metadata-phonemes.csv" \
  --data.phoneme_type text \
  --data.phonemes_path "/content/drive/MyDrive/sutta-tts-model-training/config/en[gb]_pi[si]-suttaplayer-phoneme-map.json" \
  --data.audio_dir "/content/drive/MyDrive/piper_training/wavs" \
  --data.cache_dir "/content/piper_cache" \
  --data.config_path "/content/drive/MyDrive/piper_training/en_gb-suttaplayer-medium.json" \
  --data.espeak_voice "en-gb" \
  --data.batch_size 8 \
  --ckpt_path "/content/drive/MyDrive/piper_training/last.ckpt" \
  --trainer.callbacks.class_path "train_sutta_voice.SuttaVoiceUatCallback" \
  --model.mos_metric null \
  --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 16-mixed \
  --model.sample_rate 22050 \
  --model.mel_fmin 0 \
  --model.mel_fmax 8000 \
  --model.c_mel 60 \
  --model.c_kl 1.0 \
  --data.trim_silence false
"""

# Ingest, parse, and enforce strict pre-flight gate bounds!
validator = PreFlightValidator(train_cmd)
validator.run_audit()

### Step 4: Save training command

In [9]:
with open("/content/piper_train.sh", "w") as f:
    f.write(train_cmd)
!chmod +x /content/piper_train.sh

### Step 5: Native Colab Terminal & TMUX splitting Guide
Google Colab now natively supports persistent background terminals. You can use the Terminal tab at the bottom-left of your sidebar to manage everything securely under `tmux` so that browser-reloads or dropped tabs never interrupt your session!

#### Terminal Pane Setup Sequence:

1. Open the **Terminal Tab** in the lower-left sidebar.
2. Spin up a new background terminal session:
   ```bash
   tmux new -s suttaplayer
   ```
3. **Split your pane horizontally** (or vertically) by pressing `Ctrl+B` then `"` (double quote).
4. In the **Foreground Pane (Pane 1)**, launch your validated trainer:
   ```bash
   ./piper_train.sh
   ```
5. Press `Ctrl+B` then `O` (or the arrow keys) to jump to the **Background Pane (Pane 2)** and launch your trash-free sync pruner:
   ```bash
   deno run --allow-all /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-pruner.ts
   ```
6. If you want to temporarily detach from the terminal and let it run completely independently, press `Ctrl+B` then `D`. Re-attach anytime with `tmux a -t suttaplayer`!

### Step 6: Active UAT Keep-Alive & Playback Dashboard
Run this cell in your notebook to keep your Colab session active. It synchronizes your `uat_metrics.csv` straight to your Google Sheet (`SuttaPlayer_UAT_Convergence`) every 30 seconds, and dynamically renders HTML5 audio playback widgets for your local NVMe previews—generating **exactly 0 bytes of Google Drive Trash**!

💡 **SUPPRESS BROWSER TIMEOUTS (AUTO-CLICKER JS):**
1. Press `F12` (or right-click and select Inspect) inside Brave/Chrome to open Developer Tools.
2. Click on the **Console** tab.
3. Paste the following JavaScript and press Enter:
```js
function KeepAlive() {
  let connectBtn = document.querySelector("#connect") || document.querySelector("colab-connect-button");
  if (connectBtn) {
    console.log("Simulating click on Connect Button...");
    connectBtn.click();
  }
}
setInterval(KeepAlive, 60000); // Triggers every 60 seconds
```

In [25]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Launches the Keep-Alive Sheet Sync & Local HTML5 Audio Playback Dashboard
!python3 /content/drive/MyDrive/sutta-tts-model-training/scripts/sutta-training-dashboard.py

## Tools

In [ ]:
# REPORT CHECKPOINT EPOCH

import torch
import os

# Path to the last.ckpt file
ckpt_path_to_check = f"/content/drive/MyDrive/piper_training/checkpoints/last.ckpt"  # prev

# Load the checkpoint metadata (without loading the full model weights to save memory)
# state_dict usually contains 'epoch', 'global_step', etc.
checkpoint = torch.load(ckpt_path_to_check, map_location='cpu')

# Check if it has the 'epoch' key (standard in Lightning)
if isinstance(checkpoint, dict):
    epoch = checkpoint.get('epoch', 'Unknown')
    global_step = checkpoint.get('global_step', 'Unknown')
    print(f"✅ Epoch: {epoch}")
    print(f"✅ Global Step: {global_step}")
else:
    print("⚠️  Checkpoint format unexpected. Full state dict loaded.")

### Create ONNX Model

In [ ]:
!python3 -m piper.train.export_onnx \
  --checkpoint /content/drive/MyDrive/piper_training/checkpoints/last.ckpt \
  --output-file /content/drive/MyDrive/piper_training/en_gb-suttaplayer-mediuml.onnx